# Suchtest: Stichwortsuche vs. Suche nach Bedeutung

Ziel: Bevor das Sprachmodell nur noch eine Auswahl der Berichte bekommt, prüfen wir, ob die Suche die nötigen Berichte findet. **Kein Sprachmodell, kein API-Schlüssel, keine Kosten.**

- **Daten:** 48 fiktive Berichte und 25 Testfragen (`dataset-v3`).
- **Stichwortsuche (BM25):** zählt gemeinsame Wörter; seltene Wörter zählen mehr, lange Texte werden nicht bevorzugt. Selbst geschrieben, ohne zusätzliches Paket.
- **Suche nach Bedeutung:** `intfloat/multilingual-e5-small` (MIT-Lizenz, ca. 470 MB, läuft ohne GPU) wandelt Fragen und Texte in Vektoren; verglichen wird die Ähnlichkeit.
- **Messung:** Liegen die Berichte, die für eine richtige Antwort nötig sind, unter den ersten *k* Berichten (k = 3, 5, 10)?

**Versionen**
- **s1:** Einheit ganzer Bericht; zusätzlich Kombination beider Ranglisten (war schlechter als die Suche nach Bedeutung allein und ist seit s2 entfernt).
- **s2:** Einzige Änderung: Neben dem ganzen Bericht wird auch nach **Abschnitten aus je zwei aufeinanderfolgenden Sätzen** gesucht. Ein Bericht erhält den Wert seines besten Abschnitts. Das Modell bekäme später trotzdem den ganzen Bericht. (Geplant war eine Suche nach Absätzen; die meisten Berichte bestehen aber nur aus einem Absatz.) Ergebnis: kein klarer Gewinn; seit s3 wieder entfernt (Code im Git-Verlauf, Commit `5037b2b`).
- **s3:** Einheit wieder ganzer Bericht. Einzige Änderung: **Thema erweitern.** Zu jedem der drei besten Treffer der Suche nach Bedeutung werden direkt dahinter seine ähnlichsten Berichte eingereiht (1 oder 2 Nachbarn); danach folgt die ursprüngliche Rangliste. Die Zahl der Berichte für das Modell (k) bleibt gleich, nur die Auswahl ändert sich. Ziel: alle Stände eines Themas und alle Teile einer Zusammenfassung mitliefern.

In [ ]:
!pip install -q sentence-transformers
!git clone -q https://github.com/Julien0049/AI-database.git || git -C AI-database pull -q

In [ ]:
VERSION = "s3"

import pathlib, re

DATA = pathlib.Path("AI-database/data/synthetic_reports")
reports = {p.name: p.read_text() for p in sorted(DATA.glob("AAR-*.md"))}
names = list(reports)
questions = re.findall(r"\*\*Frage\*\*\s*\n\s*> (.+)", (DATA / "test_questions.md").read_text())
print(len(reports), "Berichte,", len(questions), "Fragen")

## Nötige Berichte je Frage

Berichte, die unter den Treffern sein **müssen**, damit eine richtige Antwort möglich ist (aus `test_questions.md`). Ältere Stände zählen nur bei Fragen nach Änderungen dazu. Frage 13 hat keine nötigen Berichte (keine Antwort in den Berichten) und fließt nicht in die Kennzahlen ein.

In [ ]:
WINTER = ["AAR-2023-02-15-winter-clothing-layers.md", "AAR-2024-01-25-winter-waiting-times.md",
          "AAR-2025-12-10-winter-socks-and-jackets.md"]
NEEDED = {
    1: ["AAR-2025-01-20-shared-set-returns.md"],
    2: ["AAR-2025-01-20-shared-set-returns.md", "AAR-2025-02-04-missing-items.md"],
    3: ["AAR-2025-06-10-one-day-seminars.md"],
    4: ["AAR-2022-09-14-material-return.md"],
    5: ["AAR-2025-02-04-missing-items.md"],
    6: ["AAR-2025-02-04-missing-items.md"],
    7: ["AAR-2025-03-18-logbook-form.md"],
    8: ["AAR-2025-09-15-vehicles-multi-day-seminars.md"],
    9: ["AAR-2024-11-05-vehicle-damage.md"],
    10: ["AAR-2024-01-08-vehicle-reservations.md"],
    11: WINTER,
    12: WINTER,
    13: [],
    14: ["AAR-2023-04-12-vehicle-returns.md", "AAR-2024-01-08-vehicle-reservations.md", "AAR-2024-11-05-vehicle-damage.md",
         "AAR-2025-03-18-logbook-form.md", "AAR-2025-09-15-vehicles-multi-day-seminars.md"],
    15: WINTER,
    16: ["AAR-2025-01-20-shared-set-returns.md", "AAR-2025-09-15-vehicles-multi-day-seminars.md"] + WINTER,
    17: ["AAR-2024-03-02-multi-day-seminar-returns.md", "AAR-2025-01-20-shared-set-returns.md"],
    18: ["AAR-evening-keys-a.md"],
    19: ["AAR-evening-keys-a.md", "AAR-evening-keys-b.md"],
    20: ["AAR-2025-05-20-teamwork-day-seminar.md"],
    21: ["AAR-2025-10-07-projector-delivery.md"],
    22: ["AAR-2026-04-21-seminar-day-preparation.md"],
    23: ["AAR-2024-07-09-summer-outdoor-day.md"],
    24: ["AAR-2024-02-20-room-booking.md"],
    25: ["AAR-2026-02-03-visitor-badges.md", "AAR-2025-03-guest-wifi.md", "AAR-parking-visitors.md",
         "AAR-2026-06-16-project-planning-evening.md"],
}
assert len(NEEDED) == len(questions) and all(n in reports for v in NEEDED.values() for n in v)

## Sucheinheit: ganzer Bericht

Seit s3 wieder nur ganze Berichte (die Abschnitte aus s2 brachten keinen klaren Gewinn). `by_report` ordnet Berichte nach dem besten Wert ihrer Einheiten; bei ganzen Berichten ist das einfach ihr eigener Wert.

In [ ]:
UNITS = {"Bericht": [(n, reports[n]) for n in names]}

def by_report(units, scores):
    """Ordnet Berichte nach dem besten Wert ihrer Einheiten."""
    best = {}
    for (n, _), s in zip(units, scores):
        best[n] = max(best.get(n, float("-inf")), s)
    return sorted(best, key=lambda n: -best[n])

## Stichwortsuche (BM25)

Kleinbuchstaben, Aufteilung in Wörter, keine Grundformbildung. **Grenze:** Zusammengesetzte Wörter werden nicht zerlegt („Raumschlüssel“ passt nicht zu „Schlüssel“), Beugungen nicht vereinheitlicht („Jacke“ ≠ „Jacken“).

In [ ]:
import math
from collections import Counter

def words(text):
    return re.findall(r"[a-zäöüß0-9]+", text.lower())

def make_bm25(units, k1=1.5, b=0.75):
    """Gibt eine Suchfunktion zurück, die Berichte nach dem besten BM25-Wert ihrer Einheiten ordnet."""
    docs = [Counter(words(t)) for _, t in units]
    lengths = [sum(d.values()) for d in docs]
    avg = sum(lengths) / len(lengths)
    df = Counter(w for d in docs for w in d)
    def search(question):
        scores = []
        for d, length in zip(docs, lengths):
            s = 0.0
            for w in set(words(question)):
                if w in d:
                    idf = math.log((len(docs) - df[w] + 0.5) / (df[w] + 0.5) + 1)
                    s += idf * d[w] * (k1 + 1) / (d[w] + k1 * (1 - b + b * length / avg))
            scores.append(s)
        return by_report(units, scores)
    return search

## Suche nach Bedeutung (Embeddings)

Das Modell verlangt die Präfixe „query: “ für Fragen und „passage: “ für Berichte. Der erste Aufruf lädt das Modell (ca. 470 MB) herunter.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

def make_embed(units):
    """Gibt eine Suchfunktion zurück, die Berichte nach der höchsten Ähnlichkeit ihrer Einheiten ordnet."""
    vectors = model.encode([f"passage: {t}" for _, t in units], normalize_embeddings=True)
    def search(question):
        q = model.encode(f"query: {question}", normalize_embeddings=True)
        return by_report(units, list(vectors @ q))
    return search

## Thema erweitern (ab s3)

Ähnlichkeit zwischen Berichten = Ähnlichkeit ihrer Vektoren. Zu jedem der ersten `SEEDS` Treffer werden seine `n` ähnlichsten Berichte direkt dahinter eingereiht (sofern noch nicht enthalten); danach folgt die übrige ursprüngliche Rangliste.

In [ ]:
SEEDS = 3

report_vectors = model.encode([f"passage: {reports[n]}" for n in names], normalize_embeddings=True)
similar = {n: [names[j] for j in (report_vectors @ report_vectors[i]).argsort()[::-1] if j != i]
           for i, n in enumerate(names)}

def expand(search, n_neighbors):
    def expanded(question):
        ranking = search(question)
        out = []
        for seed in ranking[:SEEDS]:
            for r in [seed] + similar[seed][:n_neighbors]:
                if r not in out:
                    out.append(r)
        return out + [r for r in ranking if r not in out]
    return expanded

base = make_embed(UNITS["Bericht"])
METHODS = {
    "Stichwort": make_bm25(UNITS["Bericht"]),
    "Bedeutung": base,
    "Bedeutung + Thema (1)": expand(base, 1),
    "Bedeutung + Thema (2)": expand(base, 2),
}
rankings = {m: {i: f(q) for i, q in enumerate(questions, 1)} for m, f in METHODS.items()}

## Auswertung

- **Rang:** Position jedes nötigen Berichts (1 = bester Treffer).
- **vollständig@k:** Anteil der Fragen, bei denen *alle* nötigen Berichte unter den ersten k liegen. Das ist die strenge Kennzahl: Fehlt ein nötiger Bericht, kann die Antwort falsch oder unvollständig werden.

In [ ]:
KS = (3, 5, 10)
scored = [i for i in NEEDED if NEEDED[i]]

print("Rang der nötigen Berichte je Frage (" + " / ".join(METHODS) + ")\n")
for i in NEEDED:
    print(f"Frage {i}: {questions[i-1]}")
    if not NEEDED[i]:
        for m in METHODS:
            print(f"   {m:22} Top 3: {', '.join(rankings[m][i][:3])}")
        continue
    for n in NEEDED[i]:
        ranks = " / ".join(f"{rankings[m][i].index(n) + 1:>2}" for m in METHODS)
        print(f"   {ranks}   {n}")

summary = {}
print("\nvollständig@k (Anteil der", len(scored), "Fragen mit nötigen Berichten)")
for m in METHODS:
    summary[m] = {f"vollständig@{k}": round(sum(set(NEEDED[i]) <= set(rankings[m][i][:k]) for i in scored) / len(scored), 2)
                  for k in KS}
    print(f"   {m:22}", "  ".join(f"@{k}: {summary[m][f'vollständig@{k}']:.0%}" for k in KS))

## Ergebnisse speichern

In [ ]:
import json, datetime
from google.colab import files

run = {
    "versuch": f"Suchtest {VERSION}",
    "zeitpunkt": datetime.datetime.now().isoformat(timespec="seconds"),
    "embedding_modell": "intfloat/multilingual-e5-small",
    "berichte": len(reports),
    "einheiten": {u: len(v) for u, v in UNITS.items()},
    "kennzahlen": summary,
    "noetig": NEEDED,
    "ranglisten": {m: {i: r[:10] for i, r in rankings[m].items()} for m in METHODS},
}
name = f"ergebnisse_suche_{datetime.date.today()}_{VERSION}.json"
pathlib.Path(name).write_text(json.dumps(run, ensure_ascii=False, indent=2))
files.download(name)